# Experiment 6: AutoGluon time-series market trend + tabular residuals

Tree ensembles are strong at property-level interpolation but are not reliable trend extrapolators. This experiment forecasts a dense monthly market series for each `TOWN × FLAT_TYPE` segment, then uses AutoGluon Tabular to model each listing's log-rent relative to that forecast. The final prediction is `market forecast × exp(property residual)`.

The validation split is chronological: the last 12 labelled months are held out. The final forecast origin is the first test month, so no labelled rents from the forecast horizon are used by the market model.

In [1]:
# If needed: pip install autogluon.timeseries autogluon.tabular
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from autogluon.tabular import TabularPredictor
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from preprocessing import RentalPreprocessor, load_datasets

RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'
VALIDATION_HORIZON = 12
MIN_HISTORY_MONTHS = 24
TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_TIME_LIMIT', '600'))
TS_TIME_LIMIT = max(60, TIME_LIMIT_SECONDS // 2)
TABULAR_TIME_LIMIT = max(60, TIME_LIMIT_SECONDS - TS_TIME_LIMIT)


In [2]:
def canonical_flat_type(values):
    """Make `3 room` (train) and `3-room` (test) the same segment."""
    clean = values.fillna('unknown').astype(str).str.lower().str.replace(r'[^a-z0-9]+', '', regex=True)
    return clean.replace({'multigeneration': 'multigeneration'})

def add_segment_and_period(frame):
    result = frame.copy()
    result['period'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    town = result['TOWN'].fillna('unknown').astype(str).str.strip().str.lower()
    result['market_segment'] = town + ' | ' + canonical_flat_type(result['FLAT_TYPE'])
    return result

data_dir, datasets = load_datasets(project_root / 'data')
raw_train = add_segment_and_period(datasets['train'])
raw_test = add_segment_and_period(datasets['test'])
forecast_origin = raw_test['period'].min()
available_train = raw_train[raw_train['period'] < forecast_origin].copy()
if available_train.empty:
    raise ValueError('No labelled months precede the first test month.')

all_train_periods = np.sort(available_train['period'].unique())
validation_start = pd.Timestamp(all_train_periods[-VALIDATION_HORIZON])
history_raw = available_train[available_train['period'] < validation_start].copy()
validation_raw = available_train[available_train['period'] >= validation_start].copy()

print(f'All labelled history used for final forecast: {available_train.period.min():%Y-%m} to {available_train.period.max():%Y-%m}')
print(f'Forecast origin: {forecast_origin:%Y-%m}; test horizon ends: {raw_test.period.max():%Y-%m}')
print(f'Validation: train through {(validation_start - pd.offsets.MonthBegin(1)):%Y-%m}; holdout from {validation_start:%Y-%m}')


All labelled history used for final forecast: 2021-01 to 2025-02
Forecast origin: 2025-03; test horizon ends: 2026-07
Validation: train through 2024-02; holdout from 2024-03


In [3]:
def monthly_market(frame):
    monthly = (frame.groupby(['market_segment', 'period'], as_index=False)[TARGET].mean()
        .rename(columns={'market_segment': 'item_id', 'period': 'timestamp', TARGET: 'target'})
        .sort_values(['item_id', 'timestamp']))
    # Do not fit a univariate forecasting model to a brand-new sparse segment.
    eligible = monthly.groupby('item_id')['target'].count()
    return monthly[monthly['item_id'].isin(eligible[eligible >= MIN_HISTORY_MONTHS].index)]

def as_ts(monthly):
    return TimeSeriesDataFrame.from_data_frame(
        monthly, id_column='item_id', timestamp_column='timestamp'
    )

def forecast_market(history, prediction_length, model_path):
    predictor = TimeSeriesPredictor(
        target='target', prediction_length=prediction_length, freq='MS',
        eval_metric='RMSE', path=str(model_path),
    )
    predictor.fit(
        train_data=as_ts(monthly_market(history)), presets='medium_quality',
        time_limit=TS_TIME_LIMIT, random_seed=RANDOM_STATE, verbosity=2,
    )
    return predictor, predictor.predict(as_ts(monthly_market(history)))

validation_ts_predictor, validation_market_forecast = forecast_market(
    history_raw, VALIDATION_HORIZON, experiment_dir / 'ag_exp6_ts_validation'
)
display(validation_ts_predictor.leaderboard(silent=True))

validation_market = validation_market_forecast.reset_index()[['item_id', 'timestamp', 'mean']].rename(
    columns={'item_id': 'market_segment', 'timestamp': 'period', 'mean': 'market_rent_forecast'}
)
validation_with_market = validation_raw.merge(validation_market, on=['market_segment', 'period'], how='left')
print('Validation listings covered by a market forecast:', validation_with_market.market_rent_forecast.notna().mean())


Beginning AutoGluon training... Time limit = 300s
AutoGluon will save models to '/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp6_ts_validation'
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.09 GB / 16.00 GB (25.6%)
Disk Space Avail:   200.18 GB / 460.38 GB (43.5%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 12,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 

,model,score_val,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-313.558591,0.073511,0.086419,8
1,Theta,-315.640775,0.039662,0.006608,5
2,Chronos2,-366.246288,0.416109,3.971517,6
3,Toto2,-402.964206,0.012853,0.866665,7
4,RecursiveTabular,-435.415014,0.017699,0.951250,2
5,ETS,-485.648190,1.119737,0.006279,4
6,SeasonalNaive,-592.820490,1.329897,0.006960,1
7,DirectTabular,-893.716928,0.015162,1.499530,3


Validation listings covered by a market forecast: 0.99708297932597


In [4]:
def attach_observed_market_residual(frame, market_source):
    market = (market_source.groupby(['market_segment', 'period'], as_index=False)[TARGET].mean()
        .rename(columns={TARGET: 'observed_market_rent'}))
    result = frame.merge(market, on=['market_segment', 'period'], how='left')
    result['LOG_RENT_RESIDUAL'] = np.log(result[TARGET]) - np.log(result['observed_market_rent'])
    return result

def prepare_tabular_features(frame):
    result = frame.drop(columns=[TARGET, 'LOG_RENT_RESIDUAL', 'observed_market_rent', 'market_segment', 'period'], errors='ignore').copy()
    result['RENT_APPROVAL_DATE'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    return result

# The residual labels are relative to the observed market at training time.
# At inference, the observed market is replaced by the time-series forecast.
residual_train_raw = attach_observed_market_residual(history_raw, history_raw)
residual_valid_raw = attach_observed_market_residual(validation_raw, validation_raw)

validation_preprocessor = RentalPreprocessor(datasets)
residual_train = validation_preprocessor.fit_transform(residual_train_raw)
residual_valid = validation_preprocessor.transform(residual_valid_raw)
X_residual_train = prepare_tabular_features(residual_train)
X_residual_valid = prepare_tabular_features(residual_valid)

residual_predictor = TabularPredictor(
    label='LOG_RENT_RESIDUAL', problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp6_residual_validation'),
).fit(
    train_data=X_residual_train.assign(LOG_RENT_RESIDUAL=residual_train['LOG_RENT_RESIDUAL'].to_numpy()),
    tuning_data=X_residual_valid.assign(LOG_RENT_RESIDUAL=residual_valid['LOG_RENT_RESIDUAL'].to_numpy()),
    presets='medium_quality', time_limit=TABULAR_TIME_LIMIT, verbosity=2,
)

valid_residual_prediction = residual_predictor.predict(X_residual_valid).to_numpy()
validation_with_market['residual_prediction'] = valid_residual_prediction
validation_with_market['prediction'] = (
    validation_with_market['market_rent_forecast'] * np.exp(validation_with_market['residual_prediction'])
)
scored = validation_with_market.dropna(subset=['prediction'])
display(pd.DataFrame([{
    'RMSE': mean_squared_error(scored[TARGET], scored['prediction']) ** 0.5,
    'MAE': mean_absolute_error(scored[TARGET], scored['prediction']),
    'R2': r2_score(scored[TARGET], scored['prediction']),
    'Scored listings': len(scored),
}]))


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.06 GB / 16.00 GB (19.1%)
Disk Space Avail:   200.20 GB / 460.38 GB (43.5%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp6_residual_validation"
Train Data Rows:    111658
Train Data Columns: 33
Tuning Data Rows:    35310
Tuning Data Columns: 33
Label Column:       LOG_RENT_RESIDUAL
Proble

,RMSE,MAE,R2,Scored listings
0,484.419807,353.401313,0.465121,35207


In [5]:
# Refit on every labelled month strictly before the first test month.
final_preprocessor = RentalPreprocessor(datasets)
final_train_raw = attach_observed_market_residual(available_train, available_train)
final_train = final_preprocessor.fit_transform(final_train_raw)
final_test = final_preprocessor.transform(raw_test)

X_final_train = prepare_tabular_features(final_train)
X_final_test = prepare_tabular_features(final_test)
final_residual_predictor = TabularPredictor(
    label='LOG_RENT_RESIDUAL', problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp6_residual_final'),
).fit(
    train_data=X_final_train.assign(LOG_RENT_RESIDUAL=final_train['LOG_RENT_RESIDUAL'].to_numpy()),
    presets='medium_quality', time_limit=TABULAR_TIME_LIMIT, verbosity=2,
)

test_periods = pd.date_range(forecast_origin, raw_test['period'].max(), freq='MS')
final_ts_predictor, final_market_forecast = forecast_market(
    available_train, len(test_periods), experiment_dir / 'ag_exp6_ts_final'
)
test_market = final_market_forecast.reset_index()[['item_id', 'timestamp', 'mean']].rename(
    columns={'item_id': 'market_segment', 'timestamp': 'period', 'mean': 'market_rent_forecast'}
)
test_with_market = raw_test.merge(test_market, on=['market_segment', 'period'], how='left')
test_residual_prediction = final_residual_predictor.predict(X_final_test).to_numpy()
test_with_market['PREDICTED_MONTHLY_RENT'] = (
    test_with_market['market_rent_forecast'] * np.exp(test_residual_prediction)
)

# A new town/type segment cannot be forecast; use the prior global market mean.
fallback = available_train[TARGET].mean()
test_with_market['PREDICTED_MONTHLY_RENT'] = test_with_market['PREDICTED_MONTHLY_RENT'].fillna(fallback).clip(lower=0)
submission = test_with_market.drop(columns=['period', 'market_segment', 'market_rent_forecast'])
output_path = experiment_dir / 'exp_6_autogluon_timeseries_predictions.csv'
submission.to_csv(output_path, index=False)
print(f'Saved {len(submission):,} predictions to {output_path}')
display(submission[['RENT_APPROVAL_DATE', 'TOWN', 'FLAT_TYPE', 'PREDICTED_MONTHLY_RENT']].head())


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.64 GB / 16.00 GB (16.5%)
Disk Space Avail:   199.26 GB / 460.38 GB (43.3%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp6_residual_final"
Train Data Rows:    146968
Train Data Columns: 33
Label Column:       LOG_RENT_RESIDUAL
Problem Type:       regression
Preprocessing data...
Using Fea

Saved 50,000 predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_6_autogluon_timeseries_predictions.csv


,RENT_APPROVAL_DATE,TOWN,FLAT_TYPE,PREDICTED_MONTHLY_RENT
0,2025-03,kallang/whampoa,3-room,2633.212628
1,2025-03,bukit timah,4-room,3341.559796
2,2025-03,pasir ris,4-room,3109.741677
3,2025-03,punggol,4-room,3091.978602
4,2025-03,tampines,4-room,3273.141165


In [6]:
# Competition submission format: one prediction per original test row.
# Run this after the preceding cell has created `test_with_market`.
competition_submission = pd.DataFrame({
    'Id': raw_test.index.to_numpy(),
    'Predicted': test_with_market['PREDICTED_MONTHLY_RENT'].to_numpy(),
})
assert competition_submission['Id'].is_unique
assert len(competition_submission) == len(raw_test)
assert competition_submission['Predicted'].notna().all()

competition_output_path = experiment_dir / 'exp_6_autogluon_timeseries_submission.csv'
competition_submission.to_csv(competition_output_path, index=False)
print(f'Saved {len(competition_submission):,} competition-format predictions to {competition_output_path}')
display(competition_submission.head())


Saved 50,000 competition-format predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_6_autogluon_timeseries_submission.csv


,Id,Predicted
0,0,2633.212628
1,1,3341.559796
2,2,3109.741677
3,3,3091.978602
4,4,3273.141165
